# 02_Step1VsMonster_allvisits: step-1 photometry against the Monster, all visits and bands of the run

Same method as `01_Step1VsMonster` (pipeline reference selection, 1 arcsec match, `psfFlux` mag minus Monster ComCam mag), applied to every visit of the stage-1 collection.

Outputs (in `data_02_Step1VsMonster_allvisits/`):
- `matches_all.csv`: every matched star (visit, band, detector, magnitudes, colours, flag, dmag);
- `summary_per_detector.csv`: median and sigma_MAD of dmag per (visit, band, detector);
- `summary_colour_term.csv`: linear fit dmag = slope * (g - i) + offset per (visit, band) for bright stars.

Questions: is the colour slope seen in visit 2025041700536 (i band, 0.0086 mag per mag of g-i) present in the other bands and visits? Does it differ between ITL and E2V sensors? Are there detector-to-detector or visit-to-visit offsets?

Run with the `lsst` kernel at CC-IN2P3.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord
import astropy.units as u

SRC = Path('../../src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from rubincc import monster as mon

NB_TAG = '02_Step1VsMonster_allvisits'
DATA_DIR = Path(f'data_{NB_TAG}')
FIG_DIR = Path(f'figs_{NB_TAG}')
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.labelsize'] = 'x-large'
plt.rcParams['axes.titlesize'] = 'x-large'
plt.rcParams['xtick.labelsize'] = 'x-large'
plt.rcParams['ytick.labelsize'] = 'x-large'


def savefig(fig, name):
    # every figure is saved as PDF and PNG
    for ext in ('pdf', 'png'):
        fig.savefig(FIG_DIR / f'{NB_TAG}_{name}.{ext}', bbox_inches='tight', dpi=150)


def robust_stats(x):
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    return med, 1.4826 * np.median(np.abs(x - med))

In [ ]:
REPO = 'main'
USER = os.getenv('USER')
RUN_COLLECTION = f'u/{USER}/LSSTCam/runcosmos01_20261026_tract9813patch50_RunCosmosDemo/stage1'

MONSTER_DATASET = mon.DEFAULT_MONSTER_DATASET
MONSTER_COLLECTIONS = 'refcats/DM-49042/the_monster_20250219'
REF = 'monster_ComCam'

MAX_VISITS = None       # None = all visits of the selection below
SAMPLE_PER_BAND = 3     # visits per band, spread over the run (None = all visits)
MAX_SEP_ARCSEC = 0.5    # 99% of the matches of visit 2025041700536 are below 0.47 arcsec
MAG_RANGE = (16.0, 20.0)
BRIGHT_MAX = 18.5       # bright stars used for the colour term
MIN_STARS = 15          # minimum number of stars per (visit, band, detector) in the summaries

## 1. Visits of the run

In [ ]:
run_butler = mon.get_butler(REPO, collections=RUN_COLLECTION)
refs = run_butler.query_datasets('single_visit_star', limit=20000)
all_visits = sorted({ref.dataId['visit'] for ref in refs})
print(len(all_visits), 'visits with single_visit_star')

visit_info = None
try:
    records = run_butler.registry.queryDimensionRecords(
        'visit', where='instrument = inst AND visit IN (vs)', bind={'inst': 'LSSTCam', 'vs': all_visits})
    visit_info = pd.DataFrame([{
        'visit': rec.id,
        'band': rec.physical_filter.split('_')[0],
        'exp_time': getattr(rec, 'exposure_time', None),
        'zenith_angle': getattr(rec, 'zenith_angle', None),
        'day_obs': getattr(rec, 'day_obs', None),
    } for rec in records]).sort_values('visit').reset_index(drop=True)
    print(visit_info.groupby('band').size().rename('n_visits'))
    print(visit_info.head(10))
except Exception as exc:
    print('could not query the visit records:', exc)

visits_to_run = all_visits
if visit_info is not None and SAMPLE_PER_BAND is not None:
    chosen = []
    for band, g in visit_info.groupby('band'):
        vs = g['visit'].tolist()
        step = max(1, len(vs) // SAMPLE_PER_BAND)
        chosen += vs[::step][:SAMPLE_PER_BAND]
    visits_to_run = sorted(chosen)
if MAX_VISITS is not None:
    visits_to_run = visits_to_run[:MAX_VISITS]
print('visits processed in this run:', len(visits_to_run), visits_to_run)

## 2. Per-visit processing

The Monster is read shard by shard and cached, since the visits overlap the same sky area.

In [ ]:
shard_cache = {}
dropped_coord = {}  # visit -> (rows dropped for non-finite ra/dec, total rows)


def monster_for_footprint(ra_c, dec_c, radius):
    # Monster rows of all HTM7 shards touching the cone (shards are cached)
    frames = []
    for htm_id in mon.htm_ids_for_cone(ra_c, dec_c, radius):
        if htm_id not in shard_cache:
            tab = mon.load_monster_shards(run_butler, [htm_id], dataset=MONSTER_DATASET,
                                          collections=MONSTER_COLLECTIONS, verbose=False)
            if tab is None:
                shard_cache[htm_id] = None
            else:
                frame = mon.as_dataframe(tab)
                frame['ra_deg'] = mon.to_degrees(tab['coord_ra'])
                frame['dec_deg'] = mon.to_degrees(tab['coord_dec'])
                shard_cache[htm_id] = frame
        if shard_cache[htm_id] is not None:
            frames.append(shard_cache[htm_id])
    if not frames:
        raise RuntimeError(f'no Monster shard found for footprint centre ({ra_c:.3f}, {dec_c:.3f}) deg, radius {radius:.2f} deg')
    return pd.concat(frames, ignore_index=True)


def process_visit(visit):
    src_df = mon.as_dataframe(run_butler.get('single_visit_star', visit=visit, instrument='LSSTCam'))
    n_all = len(src_df)
    src_df = src_df[np.isfinite(src_df['ra']) & np.isfinite(src_df['dec'])].reset_index(drop=True)
    dropped_coord[visit] = (n_all - len(src_df), n_all)
    if len(src_df) == 0:
        raise RuntimeError(f'all {n_all} sources of visit {visit} have non-finite ra/dec')
    band = str(src_df['band'].iloc[0])

    ra_c = float(src_df['ra'].median())
    dec_c = float(src_df['dec'].median())
    pts = SkyCoord(src_df['ra'].to_numpy() * u.deg, src_df['dec'].to_numpy() * u.deg)
    radius = float(SkyCoord(ra_c * u.deg, dec_c * u.deg).separation(pts).deg.max()) + 0.05
    mon_df = monster_for_footprint(ra_c, dec_c, radius)

    # reference selection of calibrateImage (colour cut on ComCam g-r) + valid flux in this band
    mag = {b: mon.flux_njy_to_abmag(mon_df[f'{REF}_{b}_flux']) for b in 'griz'}
    gr = mag['g'] - mag['r']
    flux = mon_df[f'{REF}_{band}_flux'].to_numpy()
    sel = np.isfinite(gr) & (gr >= 0.4) & (gr <= 2.0) & (flux > 0)
    ref = mon_df[sel].reset_index(drop=True)
    gi = (mag['g'] - mag['i'])[sel]
    gaia_out = ((mon.flux_njy_to_abmag(ref['phot_g_mean_flux']) - mag['r'][sel]) > 1.0)

    idx, sep, ok = mon.crossmatch(src_df['ra'], src_df['dec'], ref['ra_deg'], ref['dec_deg'],
                                  max_sep_arcsec=MAX_SEP_ARCSEC)
    match = src_df.loc[ok, ['detector', 'psfFlux']].reset_index(drop=True)
    match['visit'] = visit
    match['band'] = band
    match['sep_arcsec'] = sep[ok]
    match['m_src'] = mon.flux_njy_to_abmag(match['psfFlux'])
    match['m_ref'] = mon.flux_njy_to_abmag(ref[f'{REF}_{band}_flux'].to_numpy()[idx[ok]])
    match['gr_ref'] = gr[sel][idx[ok]]
    match['gi_ref'] = gi[idx[ok]]
    match['flag'] = ref[f'{REF}_{band}_source_flag'].to_numpy()[idx[ok]]
    match['gaia_outlier'] = gaia_out[idx[ok]]
    match['dmag'] = match['m_src'] - match['m_ref']
    return match[np.isfinite(match['dmag'])].reset_index(drop=True)

In [ ]:
results = []
failures = []
for n, visit in enumerate(visits_to_run):
    try:
        results.append(process_visit(visit))
    except Exception as exc:
        failures.append((visit, repr(exc)))
    if (n + 1) % 10 == 0 or n + 1 == len(visits_to_run):
        print(f'{n + 1}/{len(visits_to_run)} visits processed, {len(failures)} failures, {len(shard_cache)} shards cached')

print('first failures:', failures[:5])
allm = pd.concat(results, ignore_index=True)
allm.to_csv(DATA_DIR / 'matches_all.csv', index=False)
print(allm.shape)
print(allm.groupby('band')['visit'].nunique().rename('n_visits'))

## 3. Sensor type of each detector

In [ ]:
from lsst.obs.lsst import LsstCam

camera = LsstCam().getCamera()
sensor_type = {}
for det_id in sorted(allm['detector'].unique()):
    det = camera[int(det_id)]
    sensor_type[int(det_id)] = (det.getName(), det.getPhysicalType())
allm['sensor_type'] = allm['detector'].map(lambda d: sensor_type[int(d)][1])
print(pd.DataFrame(sensor_type, index=['name', 'type']).T)

## 4. Summaries

In [ ]:
core = allm[(allm['m_ref'] >= MAG_RANGE[0]) & (allm['m_ref'] <= MAG_RANGE[1])]

rows = []
for (visit, band, det), s in core.groupby(['visit', 'band', 'detector']):
    if len(s) >= MIN_STARS:
        med, sig = robust_stats(s['dmag'])
        rows.append({'visit': visit, 'band': band, 'detector': det, 'sensor_type': s['sensor_type'].iloc[0],
                     'n': len(s), 'median_dmag': med, 'sigma_mad': sig})
per_det = pd.DataFrame(rows)
per_det.to_csv(DATA_DIR / 'summary_per_detector.csv', index=False)
print(per_det.groupby(['band', 'sensor_type'])['median_dmag'].describe().round(4))

In [ ]:
# per-visit summary for ALL processed visits, with the observing conditions
rows = []
for (visit, band), s in core.groupby(['visit', 'band']):
    med, sig = robust_stats(s['dmag'])
    rows.append({'visit': visit, 'band': band, 'n': len(s), 'median_dmag': med, 'sigma_mad': sig,
                 'frac_abs_gt_0p1': float((s['dmag'].abs() > 0.1).mean())})
per_visit_summary = pd.DataFrame(rows)
if visit_info is not None:
    per_visit_summary = per_visit_summary.merge(
        visit_info[['visit', 'exp_time', 'zenith_angle', 'day_obs']], on='visit', how='left')
per_visit_summary.to_csv(DATA_DIR / 'summary_per_visit.csv', index=False)
print(per_visit_summary.round(4).to_string(index=False))
print('visits with non-finite coordinates (rows dropped, total rows):',
      {k: v for k, v in dropped_coord.items() if v[0] > 0})

In [ ]:
# colour term of the bright stars, flag 8 excluded: dmag = slope * (g - i) + offset
# least-squares fit (sensitive to outliers) and Theil-Sen fit (robust, with 95% interval)
from scipy.stats import theilslopes
rows = []
for (visit, band), s in core.groupby(['visit', 'band']):
    b = s[(s['m_ref'] < BRIGHT_MAX) & (s['flag'] != 8)]
    if len(b) >= 30 and b['gi_ref'].std() > 0.3:
        (slope, offset), cov = np.polyfit(b['gi_ref'], b['dmag'], 1, cov=True)
        ts_slope, ts_offset, ts_lo, ts_hi = theilslopes(b['dmag'], b['gi_ref'])
        med, sig = robust_stats(b['dmag'])
        rows.append({'visit': visit, 'band': band, 'n': len(b),
                     'slope_ls': slope, 'slope_ls_err': float(np.sqrt(cov[0, 0])),
                     'slope_ts': ts_slope, 'ts_lo': ts_lo, 'ts_hi': ts_hi, 'offset_ts': ts_offset,
                     'median_dmag': med, 'sigma_mad': sig,
                     'frac_abs_gt_0p1': float((b['dmag'].abs() > 0.1).mean())})
slopes = pd.DataFrame(rows)
slopes.to_csv(DATA_DIR / 'summary_colour_term.csv', index=False)
print(slopes.round(4).to_string(index=False))
print(slopes.groupby('band')['slope_ts'].agg(['median', 'std', 'count']).round(4))

In [ ]:
# per-visit diagnostics: are the large scatter and offsets of some visits due to outliers?
print(slopes[['visit', 'band', 'n', 'median_dmag', 'sigma_mad', 'frac_abs_gt_0p1']].round(4).to_string(index=False))

visits_in = sorted(allm['visit'].unique())
ncol = 3
nrow = int(np.ceil(len(visits_in) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(6 * ncol, 4 * nrow), squeeze=False, sharex=True, sharey=True)
for ax, visit in zip(axes.ravel(), visits_in):
    s = allm[allm['visit'] == visit]
    band_v = s['band'].iloc[0]
    ax.plot(s['m_ref'], s['dmag'], '.', ms=3, alpha=0.5)
    ax.axhline(0, color='k', lw=1)
    ax.set_ylim(-0.5, 0.5)
    ax.set_title(f'{visit} ({band_v}), N={len(s)}')
    ax.set_xlabel('Monster mag')
    ax.set_ylabel('psfFlux mag - Monster mag')
savefig(fig, 'dmag_vs_mag_per_visit')

In [ ]:
# pooled colour term per band and sensor type, using only visits with a clean calibration
from scipy.stats import theilslopes

GOOD_SIGMA_MAD = 0.03
good_visits = per_visit_summary[per_visit_summary['sigma_mad'] < GOOD_SIGMA_MAD]['visit']
cg = core[core['visit'].isin(good_visits) & (core['m_ref'] < BRIGHT_MAX) & (core['flag'] != 8)]

groups = [((b, st), s) for (b, st), s in cg.groupby(['band', 'sensor_type'])]
groups += [((b, 'all'), s) for b, s in cg.groupby('band')]
rows = []
for (band, stype), s in groups:
    if len(s) >= 30 and s['gi_ref'].std() > 0.3:
        if len(s) > 3000:  # Theil-Sen is quadratic in the number of points
            s = s.sample(3000, random_state=0)
        ts_slope, ts_offset, ts_lo, ts_hi = theilslopes(s['dmag'], s['gi_ref'])
        rows.append({'band': band, 'sensor_type': stype, 'n_visits': s['visit'].nunique(), 'n_stars': len(s),
                     'slope_ts': ts_slope, 'ts_lo': ts_lo, 'ts_hi': ts_hi, 'offset_ts': ts_offset})
pooled = pd.DataFrame(rows).sort_values(['band', 'sensor_type'])
pooled.to_csv(DATA_DIR / 'summary_colour_term_pooled.csv', index=False)
print(pooled.round(4).to_string(index=False))

## 5. Figures

In [ ]:
order = {v: i for i, v in enumerate(sorted(allm['visit'].unique()))}
per_visit = core.groupby(['visit', 'band'])['dmag'].median().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for band, s in per_visit.groupby('band'):
    s = s.sort_values('visit')
    axes[0].plot(s['visit'].map(order), s['dmag'], 'o-', label=band)
axes[0].axhline(0, color='k', lw=1)
axes[0].set_xlabel('visit index (sorted)')
axes[0].set_ylabel('median(psfFlux mag - Monster mag)')
axes[0].legend()

for band, s in slopes.groupby('band'):
    s = s.sort_values('visit')
    axes[1].errorbar(s['visit'].map(order), s['slope_ts'],
                     yerr=[s['slope_ts'] - s['ts_lo'], s['ts_hi'] - s['slope_ts']], fmt='o', label=band)
axes[1].axhline(0, color='k', lw=1)
axes[1].set_xlabel('visit index (sorted)')
axes[1].set_ylabel('colour term (Theil-Sen): slope of dmag vs g - i')
axes[1].legend()
savefig(fig, 'per_visit_offsets_and_colour_term')

In [ ]:
# median offset per detector and band, labelled with the sensor type
pivot = per_det.groupby(['detector', 'band'])['median_dmag'].median().unstack()
labels = [f'{d} ({sensor_type[int(d)][1]})' for d in pivot.index]

fig, ax = plt.subplots(figsize=(14, 6))
pivot.plot.bar(ax=ax)
ax.set_xticklabels(labels, rotation=90)
ax.axhline(0, color='k', lw=1)
ax.set_ylabel('median(psfFlux mag - Monster mag)')
savefig(fig, 'per_detector_offsets')
print(pivot.round(4))